# 🌿 Detección de Zonas Deforestadas — Deep Learning
### Maestría en Inteligencia Analítica de Datos (MIAD)

**Repositorio:** [Andres-Acuna/dl_deforestation](https://github.com/Andres-Acuna/dl_deforestation)  
**Notebook:** `00_setup_and_data.ipynb`  
**Descripción:** Configuración del entorno, clonación del repositorio, montaje de Drive y descompresión del dataset.

---

## 📋 Contenido
```
Sección 1 — Verificación del entorno (GPU, versiones)
Sección 2 — Montaje de Google Drive
Sección 3 — Clonación / actualización del repositorio GitHub
Sección 4 — Instalación de dependencias
Sección 5 — Descompresión del dataset Planet Amazon
Sección 6 — Verificación de la estructura de datos
Sección 7 — Variables globales del proyecto
```

> ⚠️ **Este notebook solo se corre una vez** (o cuando se reinicie el entorno).  
> Los notebooks de entrenamiento asumen que este ya fue ejecutado correctamente.

---
## Sección 1 — Verificación del Entorno

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SECCIÓN 1 | Verificación del entorno
# ─────────────────────────────────────────────────────────────────
import subprocess, sys, os

# Verificar GPU disponible
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode == 0:
    lines = gpu_info.stdout.split('\n')
    for line in lines[:15]:
        print(line)
else:
    print('⚠️  No se detectó GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU')

# Verificar Python
print(f'\n✅ Python: {sys.version.split()[0]}')

# Verificar CUDA desde PyTorch
import torch
print(f'✅ PyTorch: {torch.__version__}')
print(f'✅ CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


---
## Sección 2 — Montaje de Google Drive

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SECCIÓN 2 | Montar Google Drive
# ─────────────────────────────────────────────────────────────────
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# Verificar que la carpeta del proyecto existe
DRIVE_BASE = '/content/drive/MyDrive/deforestation_project'

if os.path.exists(DRIVE_BASE):
    print(f'✅ Carpeta del proyecto encontrada: {DRIVE_BASE}')
    print('\nContenido:')
    for item in sorted(os.listdir(DRIVE_BASE)):
        print(f'  📁 {item}')
else:
    print(f'❌ No se encontró {DRIVE_BASE}')
    print('   Asegúrate de que la carpeta del proyecto esté creada en Drive.')
    print('   Estructura esperada:')
    print('   deforestation_project/')
    print('   ├── data/raw/planets-dataset/   ← ZIP aquí')
    print('   ├── data/processed/')
    print('   ├── checkpoints/')
    print('   ├── logs/')
    print('   └── outputs/')


---
## Sección 3 — Repositorio GitHub

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SECCIÓN 3 | Clonar o actualizar el repositorio
# ─────────────────────────────────────────────────────────────────
REPO_URL  = 'https://github.com/Andres-Acuna/dl_deforestation.git'
REPO_NAME = 'dl_deforestation'
REPO_PATH = f'/content/{REPO_NAME}'

if os.path.exists(REPO_PATH):
    print('🔄 Repositorio ya existe — actualizando...')
    result = subprocess.run(
        ['git', 'pull'],
        cwd=REPO_PATH,
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f'⚠️  Warning: {result.stderr}')
else:
    print('📥 Clonando repositorio...')
    result = subprocess.run(
        ['git', 'clone', REPO_URL, REPO_PATH],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f'✅ Repositorio clonado en {REPO_PATH}')
    else:
        print(f'❌ Error al clonar: {result.stderr}')

# Agregar el repo al path de Python para importar módulos
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
    print(f'✅ {REPO_PATH} agregado al sys.path')

# Mostrar estructura del repo
print('\nEstructura del repositorio:')
for root, dirs, files in os.walk(REPO_PATH):
    # Ignorar carpetas ocultas (.git)
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace(REPO_PATH, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in files:
        print(f'{subindent}📄 {file}')


---
## Sección 4 — Instalación de Dependencias

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SECCIÓN 4 | Instalación de librerías del proyecto
# Colab ya trae: torch, torchvision, numpy, pandas, matplotlib
# Solo instalamos las que faltan
# ─────────────────────────────────────────────────────────────────

libs = [
    'rasterio',
    'geopandas',
    'albumentations',
    'segmentation-models-pytorch',
    'earthengine-api',
]

print('📦 Instalando dependencias...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + libs,
    capture_output=True, text=True
)

if result.returncode == 0:
    print('✅ Dependencias instaladas correctamente')
else:
    print(f'⚠️  Algunos paquetes fallaron:\n{result.stderr[:500]}')

# Verificar imports críticos
print('\nVerificando imports...')
import_checks = {
    'torch':                     'PyTorch',
    'torchvision':               'TorchVision',
    'rasterio':                  'Rasterio',
    'albumentations':            'Albumentations',
    'segmentation_models_pytorch': 'SMP (U-Net)',
    'sklearn':                   'Scikit-learn',
    'numpy':                     'NumPy',
    'pandas':                    'Pandas',
    'matplotlib':                'Matplotlib',
}

all_ok = True
for module, name in import_checks.items():
    try:
        mod = __import__(module)
        version = getattr(mod, '__version__', 'n/a')
        print(f'  ✅ {name:<30} v{version}')
    except ImportError:
        print(f'  ❌ {name:<30} NO ENCONTRADO')
        all_ok = False

print('\n✅ Entorno listo' if all_ok else '\n⚠️  Revisar los paquetes marcados con ❌')


---
## Sección 5 — Descompresión del Dataset

> Esta celda detecta automáticamente si el dataset ya fue descomprimido.  
> Si ya existe la carpeta con imágenes, **no vuelve a descomprimir**.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SECCIÓN 5 | Descompresión del dataset Planet Amazon
# ─────────────────────────────────────────────────────────────────
import zipfile
from pathlib import Path

RAW_DIR    = Path(DRIVE_BASE) / 'data' / 'raw' / 'planets-dataset'
ZIP_PATH   = RAW_DIR / 'planets-dataset.zip'

# ── Verificar si ya está descomprimido ──────────────────────────
def check_already_extracted(raw_dir):
    """Busca archivos .tif o .jpg en el directorio para saber si ya se extrajo."""
    tifs = list(raw_dir.rglob('*.tif'))
    jpgs = list(raw_dir.rglob('*.jpg'))
    return len(tifs) + len(jpgs)

n_images = check_already_extracted(RAW_DIR)

if n_images > 0:
    print(f'✅ Dataset ya descomprimido — {n_images:,} imágenes encontradas en {RAW_DIR}')
    print('   Saltando descompresión.')

elif not ZIP_PATH.exists():
    print(f'❌ No se encontró el archivo ZIP en:')
    print(f'   {ZIP_PATH}')
    print('\n   Pasos para solucionarlo:')
    print('   1. Descarga el dataset desde Kaggle:')
    print('      https://www.kaggle.com/datasets/nikitarom/planets-dataset')
    print('   2. Sube el ZIP a Drive en la ruta indicada arriba.')

else:
    print(f'📦 ZIP encontrado: {ZIP_PATH}')
    print(f'   Tamaño: {ZIP_PATH.stat().st_size / 1e9:.2f} GB')
    print('\n🔄 Descomprimiendo... (esto puede tomar 3-8 minutos dependiendo de Drive)')

    # Listar contenido antes de extraer
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        all_files = z.namelist()
        print(f'   Archivos en el ZIP: {len(all_files):,}')
        print(f'   Primeros 5: {all_files[:5]}')

        # Extraer con barra de progreso simple
        total = len(all_files)
        for i, member in enumerate(all_files):
            z.extract(member, RAW_DIR)
            if i % 5000 == 0 and i > 0:
                pct = i / total * 100
                print(f'   Progreso: {i:,}/{total:,} ({pct:.1f}%)')

    n_final = check_already_extracted(RAW_DIR)
    print(f'\n✅ Descompresión completa — {n_final:,} imágenes extraídas')


---
## Sección 6 — Verificación de la Estructura de Datos

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SECCIÓN 6 | Verificación del dataset y sanity check visual
# ─────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from collections import Counter

# ── 6.1 Verificar CSV de etiquetas ──────────────────────────────
CSV_CANDIDATES = [
    RAW_DIR / 'train_v2.csv',
    RAW_DIR / 'planets-dataset' / 'train_v2.csv',
]

CSV_PATH = None
for candidate in CSV_CANDIDATES:
    if candidate.exists():
        CSV_PATH = candidate
        break

if CSV_PATH is None:
    print('❌ No se encontró train_v2.csv. Revisa la estructura del ZIP extraído.')
    print('Contenido de RAW_DIR:')
    for item in RAW_DIR.iterdir():
        print(f'  {item}')
else:
    df = pd.read_csv(CSV_PATH)
    print(f'✅ CSV cargado: {CSV_PATH}')
    print(f'   Total imágenes: {len(df):,}')
    print(f'   Columnas: {list(df.columns)}')
    print(f'\nPrimeras filas:')
    display(df.head())

    # Distribución de etiquetas
    df['tags_list'] = df['tags'].str.split()
    all_tags = [t for tags in df['tags_list'] for t in tags]
    tag_counts = Counter(all_tags)

    print('\n📊 Distribución de etiquetas:')
    for tag, count in sorted(tag_counts.items(), key=lambda x: -x[1]):
        bar = '█' * int(count / max(tag_counts.values()) * 30)
        print(f'  {tag:<25} {count:>6,}  {bar}')


In [ ]:
# ─────────────────────────────────────────────────────────────────
# SECCIÓN 6.2 | Sanity check visual — mostrar imágenes de muestra
# ─────────────────────────────────────────────────────────────────

# Buscar carpeta de TIFs
TIF_CANDIDATES = [
    RAW_DIR / 'train-tif-v2',
    RAW_DIR / 'planets-dataset' / 'train-tif-v2',
]

TIF_DIR = None
for candidate in TIF_CANDIDATES:
    if candidate.exists():
        TIF_DIR = candidate
        break

if TIF_DIR is None:
    print('⚠️  No se encontró la carpeta de TIFs. Revisa la estructura del ZIP.')
else:
    tif_files = sorted(TIF_DIR.glob('*.tif'))[:6]
    print(f'✅ Carpeta TIF: {TIF_DIR}')
    print(f'   Mostrando {len(tif_files)} imágenes de muestra...')

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    fig.suptitle('Muestra del dataset Planet Amazon — Color verdadero (RGB)',
                 fontsize=14, fontweight='bold')

    for ax, tif_path in zip(axes.flat, tif_files):
        with rasterio.open(tif_path) as src:
            # Leer bandas R(3), G(2), B(1) — orden Planet
            r = src.read(3).astype(np.float32)
            g = src.read(2).astype(np.float32)
            b = src.read(1).astype(np.float32)

            # Normalizar para visualización
            def norm(band):
                p2, p98 = np.percentile(band, [2, 98])
                return np.clip((band - p2) / (p98 - p2 + 1e-6), 0, 1)

            rgb = np.stack([norm(r), norm(g), norm(b)], axis=-1)

        # Etiqueta de la imagen
        img_name = tif_path.stem
        if CSV_PATH is not None:
            row = df[df['image_name'] == img_name]
            label = row['tags'].values[0] if len(row) > 0 else 'N/A'
        else:
            label = img_name

        ax.imshow(rgb)
        ax.set_title(label, fontsize=8, wrap=True)
        ax.axis('off')

    plt.tight_layout()
    plt.savefig(
        Path(DRIVE_BASE) / 'outputs' / 'sample_images.png',
        dpi=120, bbox_inches='tight'
    )
    plt.show()
    print('✅ Muestra guardada en outputs/sample_images.png')


In [ ]:
# ─────────────────────────────────────────────────────────────────
# SECCIÓN 6.3 | Verificar metadatos de un TIF
# ─────────────────────────────────────────────────────────────────

if TIF_DIR is not None:
    sample_tif = sorted(TIF_DIR.glob('*.tif'))[0]

    with rasterio.open(sample_tif) as src:
        print(f'📡 Metadatos de: {sample_tif.name}')
        print(f'   Número de bandas : {src.count}  (esperado: 4)')
        print(f'   Dimensiones      : {src.height} x {src.width} px  (esperado: 256 x 256)')
        print(f'   Tipo de dato     : {src.dtypes[0]}')
        print(f'   CRS              : {src.crs}')
        print(f'   Transform        : {src.transform}')

        # Estadísticas por banda
        print('\n   Estadísticas por banda:')
        band_names = ['Azul (B)', 'Verde (G)', 'Rojo (R)', 'NIR']
        for i, name in enumerate(band_names, start=1):
            data = src.read(i).astype(np.float32)
            print(f'   Banda {i} {name:<12} '
                  f'min={data.min():.0f}  '
                  f'max={data.max():.0f}  '
                  f'mean={data.mean():.0f}  '
                  f'std={data.std():.0f}')

    if src.count == 4 and src.height == 256:
        print('\n✅ Formato correcto — el pipeline de datos puede continuar')
    else:
        print('\n⚠️  El formato no coincide con lo esperado. Revisar el dataset descargado.')


---
## Sección 7 — Variables Globales del Proyecto

> Ejecuta esta celda al inicio de **cada notebook** del proyecto.  
> Define todas las rutas y configuraciones centralizadas.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# SECCIÓN 7 | Configuración centralizada del proyecto
# Copiar este bloque al inicio de cada notebook del proyecto
# ─────────────────────────────────────────────────────────────────
import os, sys
from pathlib import Path

# ── Rutas base ───────────────────────────────────────────────────
DRIVE_BASE   = Path('/content/drive/MyDrive/deforestation_project')
REPO_PATH    = Path('/content/dl_deforestation')

# ── Datos ────────────────────────────────────────────────────────
DATA_DIR      = DRIVE_BASE / 'data'
RAW_PLANET    = DATA_DIR / 'raw' / 'planets-dataset'
RAW_COLOMBIA  = DATA_DIR / 'raw' / 'colombia'
PROC_PLANET   = DATA_DIR / 'processed' / 'planet'
PROC_COLOMBIA = DATA_DIR / 'processed' / 'colombia'
MASKS_DIR     = DATA_DIR / 'masks'

# ── Artefactos del modelo ────────────────────────────────────────
CKPT_DIR      = DRIVE_BASE / 'checkpoints'
LOGS_DIR      = DRIVE_BASE / 'logs'
OUTPUTS_DIR   = DRIVE_BASE / 'outputs'

# ── Archivos específicos ─────────────────────────────────────────
PLANET_CSV    = RAW_PLANET / 'train_v2.csv'
PLANET_TIF    = RAW_PLANET / 'train-tif-v2'
BEST_CKPT     = CKPT_DIR / 'planet_best.pt'
COLOMBIA_CKPT = CKPT_DIR / 'colombia_best.pt'

# ── Hiperparámetros ──────────────────────────────────────────────
CONFIG = {
    # Datos
    'img_size'      : 224,
    'batch_size'    : 32,
    'num_workers'   : 2,
    'tile_size'     : 224,
    'tile_stride'   : 112,
    # Fase 1 — Planet
    'planet_epochs' : 20,
    'planet_lr'     : 1e-3,
    'num_classes'   : 17,
    # Fase 2 — Colombia
    'colombia_epochs': 30,
    'colombia_lr'    : 5e-4,
    # Reproducibilidad
    'seed'          : 42,
}

# ── Etiquetas Planet Amazon ───────────────────────────────────────
PLANET_TAGS = [
    'agriculture', 'artisinal_mine', 'bare_ground', 'blooming',
    'blow_down', 'clear', 'cloudy', 'conventional_mine', 'cultivation',
    'habitation', 'haze', 'partly_cloudy', 'primary', 'road',
    'selective_logging', 'slash_burn', 'water'
]
DEFORESTATION_TAGS = {
    'agriculture', 'slash_burn', 'habitation',
    'cultivation', 'bare_ground', 'artisinal_mine',
    'selective_logging', 'blow_down'
}

# ── Verificación de rutas ─────────────────────────────────────────
import torch
torch.manual_seed(CONFIG['seed'])

print('📁 Rutas del proyecto:')
paths = {
    'Drive base'    : DRIVE_BASE,
    'Repositorio'   : REPO_PATH,
    'Planet raw'    : RAW_PLANET,
    'Colombia raw'  : RAW_COLOMBIA,
    'Planet proc.'  : PROC_PLANET,
    'Colombia proc.': PROC_COLOMBIA,
    'Máscaras'      : MASKS_DIR,
    'Checkpoints'   : CKPT_DIR,
    'Logs'          : LOGS_DIR,
    'Outputs'       : OUTPUTS_DIR,
}
for name, path in paths.items():
    status = '✅' if Path(path).exists() else '❌'
    print(f'  {status} {name:<18} {path}')

print(f'\n🖥️  Dispositivo: {"GPU — " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'🔢 Seed fijada: {CONFIG["seed"]}')
print('\n✅ Configuración lista — puedes continuar con el siguiente notebook')


---
## ✅ Setup Completado

Si todas las secciones corrieron sin errores `❌`, el entorno está listo.  
El siguiente paso es abrir el notebook de entrenamiento:

```
notebooks/
├── 00_setup_and_data.ipynb     ← este notebook
├── 01_eda_planet.ipynb          ← análisis exploratorio
├── 02_train_planet.ipynb        ← Fase 1: entrenamiento backbone
└── 03_finetune_colombia.ipynb   ← Fase 2: fine-tuning Colombia
```

### Checklist antes de continuar
- [ ] GPU activa (Sección 1)
- [ ] Drive montado correctamente (Sección 2)
- [ ] Repositorio clonado (Sección 3)
- [ ] Todas las dependencias instaladas (Sección 4)
- [ ] Dataset descomprimido con imágenes visibles (Sección 5)
- [ ] TIFs con 4 bandas y 256×256 px confirmados (Sección 6)
- [ ] Variables globales sin rutas en `❌` (Sección 7)